In [5]:
from math import sqrt
from numpy import array, linspace
from matplotlib import pyplot as plt
import matplotlib
from scipy import stats
from pyproj import Transformer

In [18]:
# Fixpunkt 6420 (E,N,H)

FixPoint_nrcanPos_UTM = array([382715.572, 7427154.929, 61.803])    # <-- indsæt dine værdier fra NRCAN i UTM zone 22 her (# ITRF2020/IGc20 (epoch 2026.7))
FixPoint_true_UTM = array([382716.145, 7427154.542, 61.838])  # UTM zone 22 ,UTM22 GR96

pipeline = "+ellps=GRS80 +proj=pipeline +step +proj=utm +zone=22 +inv +step +proj=cart"
transform_object = Transformer.from_pipeline(pipeline)

FixPoint_nrcanPos_XYZ = transform_object.transform(*FixPoint_nrcanPos_UTM)
FixPoint_true_XYZ = transform_object.transform(*FixPoint_true_UTM)

print(FixPoint_nrcanPos_XYZ)

gr96_time = 1996.623
obs_time = 2026 + (261/365) # <--- Observationstid

# flyt det Nord Amerikanske kontinent tilbage til 1996.623 (stadig i ITR2020) 
pipeline = (f"+ellps=GRS80 +proj=pipeline +step +inv +init=ITRF2020:NOAM +t_epoch={gr96_time}")
trans_gr96_t = Transformer.from_pipeline(pipeline)

FixPoint_nrcanPos_XYZ_GR96 = trans_gr96_t.transform(*FixPoint_nrcanPos_XYZ)

print(FixPoint_nrcanPos_XYZ_GR96)

# transformer fra ITRF2020 til ITRF94 i 1996.623
pipeline = ("+ellps=GRS80 +proj=pipeline " +
            f"+step +init=ITRF2020:ITRF94 +t_obs={gr96_time}")
trans_itrf = Transformer.from_pipeline(pipeline)

FixPoint_nrcanPos_XYZ_GR96_ITRF94 = trans_itrf.transform(*FixPoint_nrcanPos_XYZ_GR96)

print(FixPoint_nrcanPos_XYZ_GR96_ITRF94)
print(FixPoint_true_XYZ)

import numpy as np
print(np.linalg.norm(np.array([FixPoint_nrcanPos_XYZ_GR96_ITRF94-FixPoint_true_XYZ])))



(1483843.782525203, -2018816.7266166105, 5845827.174817731)
(1483843.782525203, -2018816.7266166105, 5845827.174817731)
(1483843.7984544013, -2018816.7359617052, 5845827.1201841235)
(1483844.4627479813, -2018816.6570692775, 5845827.065209505)


TypeError: unsupported operand type(s) for -: 'tuple' and 'tuple'

In [49]:
import numpy as np
from pyproj import Transformer

# ---------------------------------------------------------------------
# Input coordinates
# ---------------------------------------------------------------------

# NRCAN position:
# ITRF2020 / IGc20, epoch 2026.7, UTM zone 22N
fixpoint_nrcan_utm = np.array([
    382715.572,
    7427154.929,
    61.803,
])

# Reference position:
# GR96 / UTM zone 22N
fixpoint_true_utm = np.array([
    382716.145,
    7427154.542,
    61.838,
])

lat = 66 + 56/60 + 21.91368/3600
lon = -(53 + 41/60 + 1.91414/3600)
h = 61.803

fixpoint_nrcan_latlon = np.array([lon, lat, h])

# ---------------------------------------------------------------------
# Difference in UTM coordinates
# ---------------------------------------------------------------------

utm_difference = fixpoint_nrcan_utm - fixpoint_true_utm

print("Difference UTM [m]:")
print(f"  Easting : {utm_difference[0]:.3f}")
print(f"  Northing: {utm_difference[1]:.3f}")
print(f"  Height  : {utm_difference[2]:.3f}")
print(f"  3D error: {np.linalg.norm(utm_difference):.3f} m")

# ---------------------------------------------------------------------
# NRCAN geographic coordinates
# ---------------------------------------------------------------------

lat = 66 + 56/60 + 21.91368/3600
lon = -(53 + 41/60 + 1.91414/3600)
h = 61.803

# pyproj with always_xy=True expects:
# longitude, latitude, ellipsoidal height
fixpoint_nrcan_latlonh = np.array([lon, lat, h])

print("NRCAN lon, lat, h:")
print(fixpoint_nrcan_latlonh)

# ---------------------------------------------------------------------
# ITRF2020 geographic -> ITRF2020 geocentric XYZ
# ---------------------------------------------------------------------

transformer = Transformer.from_crs(
    "EPSG:9989",   # ITRF2020 geographic 3D
    "EPSG:9988",   # ITRF2020 geocentric XYZ
    always_xy=True,
)

X, Y, Z = transformer.transform(
    lon,
    lat,
    h,
)

nrcan_xyz = np.array([X, Y, Z])

print("\nITRF2020 XYZ:")
print(nrcan_xyz)

# ---------------------------------------------------------------------
# Transform ITRF2020 coordinates to ITRF94
# ---------------------------------------------------------------------

# Epochs
obs_time = 2026 + 261 / 365
gr96_time = 1996.623


itrf2020_to_itrf94 = Transformer.from_pipeline(
    "+proj=pipeline "
    "+step +proj=helmert +convention=coordinate_frame "
    f"+t_obs={obs_time}"
)

nrcan_itrf94_xyz = np.array(
    itrf2020_to_itrf94.transform(*nrcan_xyz)
)

# ---------------------------------------------------------------------
# Convert GR96 / UTM22 reference coordinates to geocentric XYZ
# ---------------------------------------------------------------------

gr96_to_xyz = Transformer.from_crs(
    "EPSG:3182",   # UTM zone 22N
    "EPSG:4978",   # GRS1980 geocentric XYZ
    always_xy=True,
)

true_xyz = np.array(
    gr96_to_xyz.transform(*fixpoint_true_utm)
)

# ---------------------------------------------------------------------
# Compare XYZ coordinates
# ---------------------------------------------------------------------

xyz_difference = nrcan_itrf94_xyz - true_xyz

print("\nXYZ coordinates:")
print("NRCAN:", nrcan_itrf94_xyz)
print("True :", true_xyz)

print("\nDifference XYZ [m]:")
print(f"  X: {xyz_difference[0]:.3f}")
print(f"  Y: {xyz_difference[1]:.3f}")
print(f"  Z: {xyz_difference[2]:.3f}")

print(f"\n3D error [m]: {np.linalg.norm(xyz_difference):.3f}")

Difference UTM [m]:
  Easting : -0.573
  Northing: 0.387
  Height  : -0.035
  3D error: 0.692 m
NRCAN lon, lat, h:
[-53.68386504  66.93942047  61.803     ]

ITRF2020 XYZ:
[ 1483843.78246889 -2018816.72701646  5845827.17469477]

XYZ coordinates:
NRCAN: [ 1483843.78246889 -2018816.72701646  5845827.17469477]
True : [ 1483844.46272729 -2018816.65704112  5845827.06532078]

Difference XYZ [m]:
  X: -0.680
  Y: -0.070
  Z: 0.109

3D error [m]: 0.693


In [71]:
import numpy as np


# Measured 2026 DOY 261 (# ITRF2020 / IGc20, epoch 2026.7, UTM zone 22N)
fixpoint_nrcan_utm = np.array([
    382715.572,   # E [m]
    7427154.929,  # N [m]
    61.803,       # U [m]
])

pipeline = "+ellps=GRS80 +proj=pipeline +step +proj=utm +zone=22 +inv +step +proj=cart"
transform_object = Transformer.from_pipeline(pipeline)

ITRF_2020_nrcan = transform_object.transform(*fixpoint_nrcan_utm) # XYZ ITRF2020 9988
print(ITRF_2020_nrcan)

transform_9988_to_10954 = Transformer.from_crs(
    "EPSG:9988",   # ITRF2020
    "EPSG:10956",   # EPSG:10954
    always_xy=True,
)

GR96_1996_nrcan = transform_object.transform(*ITRF_2020_nrcan) # XYZ ITRF2020 9988
print(GR96_1996_nrcan )


gr96_time = 1996.623

# Reference position:
# GR96 / UTM zone 22N, measured 2006
fixpoint_true_utm = np.array([
    382716.145,
    7427154.542,
    61.838,
])

# Velocity vector [E, N, U]
velocity = np.array([
    -18.6,  # mm/year
     11.6,  # mm/year
      2.9,  # mm/year
]) / 1000.0  # -> m/year

# Epochs
doy = 261
t_2026 = 2026 + (doy - 1) / 365.0
gr96_time = 1996.623

dt = t_2026 - gr96_time

# Propagate 2026 measurement backwards to 2006
fixpoint_2006 = fixpoint_nrcan_utm - velocity * dt

print(f"2026 epoch: {t_2026:.6f}")
print(f"Time difference: {dt:.6f} years")

print("\nPropagated 2026 position to 2006:")
print(f"E = {fixpoint_2006[0]:.3f} m")
print(f"N = {fixpoint_2006[1]:.3f} m")
print(f"U = {fixpoint_2006[2]:.3f} m")

print("\nDifference from 2006 reference:")
difference = fixpoint_2006 - fixpoint_true_utm
print(np.linalg.norm(np.array(difference)))

print(f"dE = {difference[0]:+.3f} m")
print(f"dN = {difference[1]:+.3f} m")
print(f"dU = {difference[2]:+.3f} m")

(1483843.782525203, -2018816.7266166105, 5845827.174817731)


CRSError: Invalid projection: EPSG:10956: (Internal Proj Error: proj_create: crs not found: EPSG:10956)